# Biome Source Generation Helper

## Constants

In [127]:
# --------------------------------------------------------------------
# 1.  Pre-defined vanilla-style label → numeric range tables
# --------------------------------------------------------------------
LABEL_RANGES = {
    "depth": [ # 0-4
        (-0.005, 0.2),     # surface slice (up to sea level)
        (0.2, 0.35),      # shallow caves
        (0.35, 0.5),      # mid-depth, below which is deepslate
        (0.5, 0.75),      # deep layer
        (0.75, 1.0)       # deepest layer
    ],
    "PV": [ # 0-4
        (-1.0, -0.85),   # valleys
        (-0.85, -0.6),   # low
        (-0.6, 0.2),     # mid
        (0.2, 0.7),      # high
        (0.7, 1.0),      # peaks
    ],
    "erosion": [ # 0-6
        (-1.0, -0.78),
        (-0.78, -0.375),
        (-0.375, -0.2225),
        (-0.2225, 0.05),
        (0.05, 0.45),
        (0.45, 0.55),
        (0.55, 1.0),
    ],
    "continentalness": [ # 0-8
        (-1.2, -1.05),       # mushroom fields
        (-1.05, -0.455),     # deep ocean
        (-0.455, -0.19),     # ocean
        (-0.19, -0.11),      # coast
        (-0.11, 0.03),       # near-inland
        (0.03, 0.3),         # mid-inland
        (0.3, 0.6),          # far inland
        (0.6, 0.8),          # very far inland
        (0.8, 1.0),          # extreme far land
    ],
    # "continentalness_new": [ # 0-8
    #     (-1.2, -1.05),       # mushroom fields
    #     (-1.05, -0.255),     # deep ocean
    #     (-0.255, -0.0),     # ocean
    #     (-0.0, 0.1),      # coast
    #     (0.1, 0.2),       # near-inland
    #     (0.2, 0.4),         # mid-inland
    #     (0.4, 0.6),          # far inland
    #     (0.6, 0.8),          # very far inland
    #     (0.8, 1.0),          # extreme far land
    # ],
    "temperature": [ # 0-6
        (-1.0, -0.45),
        (-0.45, -0.15),
        (-0.15, 0.2),
        (0.2, 0.35),
        (0.35, 0.55),
        (0.55, 0.75),
        (0.75, 1.0),
    ],
    "humidity": [ # 0-6
        (-1.0, -0.35),
        (-0.35, -0.1),
        (-0.1, 0.1),
        (0.1, 0.3),
        (0.3, 0.6),
        (0.6, 0.8),
        (0.8, 1.0),
    ],
    "weirdness": [ # 0-1
        (-1.0, 0.0),
        (0.0, 1.0)
    ],
}

PARAM_ORDER = ["depth", "PV", "erosion", "continentalness",
               "temperature", "humidity", "weirdness"]

## Reading from a list

In [136]:
# --------------------------------------------------------------------
# 2.  Conversion helpers
# --------------------------------------------------------------------
# ------------------------------------------------------------------
# 2.  Helpers to interpret “label specs”
# ------------------------------------------------------------------
def _expand_label_spec(param: str, spec_value):
    """
    Acceptable forms:
      * missing / None           → all labels for that param
      * int                      → just that label
      * list/tuple of ints       → explicit set
      * 2-tuple (a,b)            → inclusive range a…b
    """
    n_labels = len(LABEL_RANGES[param])

    if spec_value is None:
        return list(range(n_labels))

    # scalar int
    if isinstance(spec_value, int):
        return [spec_value]

    # 2-tuple treated as range
    if isinstance(spec_value, tuple) and len(spec_value) == 2:
        lo, hi = spec_value
        if lo > hi:
            lo, hi = hi, lo
        return list(range(lo, hi + 1))

    # generic list/tuple
    if isinstance(spec_value, (list, tuple)):
        return list(spec_value)

    raise TypeError(f"Bad label spec {spec_value!r} for param {param}")

def _labels_to_interval(param: str, labels):
    lows  = [LABEL_RANGES[param][i][0] for i in labels]
    highs = [LABEL_RANGES[param][i][1] for i in labels]
    return [min(lows), max(highs)]

def convert_biome_specs(spec: dict) -> dict:
    """
    Convert the user-friendly label spec into a Minecraft
    multi_noise JSON blob (normal.json structure).
    """
    rows = []
    for biome_id, params in spec.items():
        p_out = {}
        for p in LABEL_RANGES:
            labels = _expand_label_spec(p, params.get(p))
            p_out[p] = _labels_to_interval(p, labels)
        if params.get("offset") is not None:
            p_out["offset"] = params["offset"]
        else:
            p_out["offset"] = 0
        rows.append({"biome": biome_id, "parameters": p_out})

    normal_json = {
        "dimensions": {
            "minecraft:overworld": {
                "type": "minecraft:overworld",
                "generator": {
                    "type": "minecraft:noise",
                    "biome_source": {
                        "type": "minecraft:multi_noise",
                        "biomes": rows
                    },
                    "settings": "minecraft:overworld"
                }
            },
            "minecraft:the_end": {
                "type": "minecraft:the_end",
                "generator": {
                    "type": "minecraft:noise",
                    "biome_source": {
                    "type": "minecraft:the_end"
                    },
                    "settings": "minecraft:end"
                }
            },
            "minecraft:the_nether": {
                "type": "minecraft:the_nether",
                "generator": {
                    "type": "minecraft:noise",
                    "biome_source": {
                    "type": "minecraft:multi_noise",
                    "preset": "minecraft:nether"
                    },
                    "settings": "minecraft:nether"
                }
            }
        }
    }
    return normal_json


### Testing

In [137]:
list_example_spec = [
    {
        "biome": "minecraft:deep_frozen_ocean",
        "params": {
            "continentalness": 1
        }
        
    },
    {
        "biome": "minecraft:frozen_ocean",
        "params": {
            "continentalness": 2
        }
        
    }
]

preset_json = convert_biome_specs(list_example_spec)
out_path = "list_example.json"
with open(out_path, "w") as f:
    json.dump(preset_json, f, indent=2)
print("Wrote", out_path)

AttributeError: 'list' object has no attribute 'items'

## Reading from a tree

In [ ]:
import json
from copy import deepcopy
import numpy as np


# -----------------------------------------------------------
#  Tree parsing
# -----------------------------------------------------------
def tree_to_spec(tree):
    spec = []

    def dfs(node, depth_idx, acc_params):
        if isinstance(node, dict) and "biome" in node:
            biome_id = node["biome"]
            # just append the biome with the current params to the spec
            biome_dict = {
                "biome": biome_id,
                "params": {}
            }
            
            for k, v in acc_params.items():
                if v is not None:
                    biome_dict["params"][k] = v
            spec.append(biome_dict)
            return

        if depth_idx >= len(PARAM_ORDER):
            raise ValueError("Tree deeper than expected param order")

        param_key = PARAM_ORDER[depth_idx]

        for key, child in node.items():
            if key == "*":
                next_params = deepcopy(acc_params)
                # no constraint for this param
                dfs(child, depth_idx + 1, next_params)
                continue

            # interpret key as label spec
            if isinstance(key, tuple) and len(key) == 2:
                labels = list(range(min(key), max(key) + 1))
            elif isinstance(key, int):
                labels = [key]
            elif isinstance(key, (list, set)):
                labels = list(key)
            else:
                raise TypeError(f"Unsupported key type: {key}")

            next_params = deepcopy(acc_params)
            next_params[param_key] = labels
            dfs(child, depth_idx + 1, next_params)

    dfs(tree, 0, {})
    return spec

# -----------------------------------------------------------
#  Interval conversion helpers from previous answers
# -----------------------------------------------------------
def _expand(param, value):
    n = len(LABEL_RANGES[param])
    if value is None:
        return range(n)
    if isinstance(value, int):
        return [value]
    if isinstance(value, tuple) and len(value) == 2:
        lo, hi = sorted(value)
        return range(lo, hi + 1)
    if isinstance(value, (list, set, tuple)):
        return list(value)
    raise TypeError(f"Bad spec {value!r} for {param}")


def _w_to_pv(w):
    pv = 1 - np.abs(3 * np.abs(w) - 2)
    return pv

def _w_to_pv_intervals(intervals):
    for interval in intervals:
        w_interval = [_w_to_pv(interval[0]), _w_to_pv(interval[1])]
        print(f"w: {interval} -> pv: {w_interval}")

# this is a multi-valued function
# it returns a list of intervals
# that are the possible values of w
# given the input interval of pv
# the intervals are in the range [-1, 1]
# w_sign is a sign restriction for w
# it can be either -1 or 1 or 0
# if w_sign is -1, then the function will return
# the intervals of w that are in the range [-1, 0]
# if w_sign is 1, then the function will return
# the intervals of w that are in the range [0, 1]
# if w_sign is 0, then the function will return
# the intervals of w that are in the range [-1, 1]
def _pv_to_w(interval, w_sign=0):
    all_pv_intervals = []
    
    p_low = interval[0]
    p_high = interval[1]
    
    # assert that interval is in the range [-1, 1] and low is less than high
    if p_low < -1 or p_high > 1 or p_low > p_high:
        raise ValueError("Interval must be in the range [-1, 1] and low must be less than high")
    
    u_low = 1 - p_high
    u_high = 1 - p_low
    
    # both in section I
    if u_low >= 1 and u_low <= 2 and u_high >= 1 and u_high <= 2:
        w_a_low = (u_low - 2) / 3
        w_a_high = (u_high - 2) / 3
        w_c_low = (u_high - 2) / (-3)
        w_c_high = (u_low - 2) / (-3)
        if w_sign <= 0:
            all_pv_intervals.append([w_a_low, w_a_high])
        if w_sign >= 0:
            all_pv_intervals.append([w_c_low, w_c_high])
        return all_pv_intervals
    # both in section II
    if u_low >= 0 and u_low <= 1 and u_high >= 0 and u_high <= 1:
        w_a_low = (u_low - 2) / 3
        w_a_high = (u_high - 2) / 3
        w_c_low = (u_high - 2) / (-3)
        w_c_high = (u_low - 2) / (-3)
        
        w_d_low = (u_low + 2) / 3
        w_d_high = (u_high + 2) / 3
        w_b_low = (u_high + 2) / (-3)
        w_b_high = (u_low + 2) / (-3)
            
        if w_sign <= 0:
            all_pv_intervals.append([w_a_low, w_a_high])
            all_pv_intervals.append([w_b_low, w_b_high])
        if w_sign >= 0:
            all_pv_intervals.append([w_c_low, w_c_high])
            all_pv_intervals.append([w_d_low, w_d_high])
        return all_pv_intervals
    # u_high in section I, u_low in section II
    if u_high >= 1 and u_high <= 2 and u_low >= 0 and u_low <= 1:
        # essentially, this can be reduced to the previous two cases, introducing a midpoint at u=1
        u_mid = 1 # this is the "low" for u_high, the "high" for u_low
        # reduce to section I
        w_a_low_I = (u_mid - 2) / 3
        w_a_high_I = (u_high - 2) / 3
        w_c_low_I = (u_high - 2) / (-3)
        w_c_high_I = (u_mid - 2) / (-3)
        
        # reduce to section II
        w_a_low_II = (u_low - 2) / 3
        w_a_high_II = (u_mid - 2) / 3
        w_c_low_II = (u_mid - 2) / (-3)
        w_c_high_II = (u_low - 2) / (-3)
        
        
        w_d_low_II = (u_low + 2) / 3
        w_d_high_II = (u_mid + 2) / 3
        w_b_low_II = (u_mid + 2) / (-3)
        w_b_high_II = (u_low + 2) / (-3)
        
        if w_sign <= 0:
            all_pv_intervals.append([w_a_low_I, w_a_high_I])
            all_pv_intervals.append([w_a_low_II, w_a_high_II])
            all_pv_intervals.append([w_b_low_II, w_b_high_II])
        if w_sign >= 0:
            all_pv_intervals.append([w_c_low_I, w_c_high_I])
            all_pv_intervals.append([w_c_low_II, w_c_high_II])
            all_pv_intervals.append([w_d_low_II, w_d_high_II])
        return all_pv_intervals
    

def simplify_intervals(intervals, eps=1e-5):
    """
    Simplify the intervals by merging overlapping ones
    and removing small gaps between them.
    """
    intervals = sorted(intervals, key=lambda x: x[0])
    simplified = []
    current = intervals[0]
    
    for next_interval in intervals[1:]:
        if current[1] + eps >= next_interval[0]:
            current[1] = max(current[1], next_interval[1])
        else:
            simplified.append(current)
            current = next_interval
    simplified.append(current)
    
    rounded_intervals = round_intervals(simplified, 4)
    
    return rounded_intervals


def round_intervals(intervals, precision=4):
    """
    Round the intervals to the nearest 0.0001
    and remove small gaps between them.
    """
    rounded_intervals = []
    for interval in intervals:
        low = round(interval[0], precision)
        high = round(interval[1], precision)
        if low > high:
            low, high = high, low
        rounded_intervals.append([low, high])
    
    return rounded_intervals
        
    
def _interval(param, labels):
    lows  = [LABEL_RANGES[param][i][0] for i in labels]
    highs = [LABEL_RANGES[param][i][1] for i in labels]
    interval = [min(lows), max(highs)]
    return interval

def convert(spec):
    rows = []
    for biome_dict in spec:
        biome = biome_dict["biome"]
        params = biome_dict["params"]
        
        weirdness_labels = _expand("weirdness", params.get("weirdness"))
        if len(weirdness_labels) == 2:
            # this means no sign restriction
            sign_of_weirdness = 0
        elif len(weirdness_labels) == 1:
            # map weirdness_labels[0] 0 or 1 to -1 or 1
            if weirdness_labels[0] == 0:
                sign_of_weirdness = -1
            elif weirdness_labels[0] == 1:
                sign_of_weirdness = 1
            else:
                raise ValueError("Weirdness must be a single value a list of two values: 0 or 1")
        else:
            raise ValueError("Weirdness must be a single value a list of two values: 0 or 1")
        pv_interval = _interval("PV", _expand("PV", params.get("PV")))
        w_intervals = simplify_intervals(_pv_to_w(pv_interval, w_sign=sign_of_weirdness))
        
        
        for w_interval in w_intervals:
            pjson = {}
            pjson["weirdness"] = w_interval
            for p in LABEL_RANGES.keys():
                if p == "PV" or p == "weirdness":
                    continue
                interval = _interval(p, _expand(p, params.get(p)))
                pjson[p] = interval
            if params.get("offset") is not None:
                pjson["offset"] = params["offset"]
            else:
                pjson["offset"] = 0
            rows.append({"biome": biome, "parameters": pjson})
    return rows

def to_preset_file(biome_rows):
    return {
        "dimensions": {
            "minecraft:overworld": {
                "type": "minecraft:overworld",
                "generator": {
                    "type": "minecraft:noise",
                    "biome_source": {
                        "type": "minecraft:multi_noise",
                        "biomes": biome_rows
                    },
                    "settings": "minecraft:overworld"
                }
            },
            "minecraft:the_end": {
                "type": "minecraft:the_end",
                "generator": {
                    "type": "minecraft:noise",
                    "biome_source": {
                    "type": "minecraft:the_end"
                    },
                    "settings": "minecraft:end"
                }
            },
            "minecraft:the_nether": {
                "type": "minecraft:the_nether",
                "generator": {
                    "type": "minecraft:noise",
                    "biome_source": {
                    "type": "minecraft:multi_noise",
                    "preset": "minecraft:nether"
                    },
                    "settings": "minecraft:nether"
                }
            }
        }
    }

### Testing

In [77]:
p_from_u = lambda x: 1 - x

_pv_to_w([p_from_u(1),p_from_u(0)])

[[-0.6666666666666666, -0.3333333333333333],
 [-1.0, -0.6666666666666666],
 [0.3333333333333333, 0.6666666666666666],
 [0.6666666666666666, 1.0]]

In [78]:
_pv_to_w(LABEL_RANGES["PV"][1], w_sign=0)

[[-0.1333333333333333, -0.04999999999999997],
 [0.04999999999999997, 0.1333333333333333]]

In [79]:
simplify_intervals(_pv_to_w(LABEL_RANGES["PV"][1], w_sign=0))

[[-0.1333, -0.05], [0.05, 0.1333]]

In [80]:
_pv_to_w([p_from_u(1),p_from_u(0)])
_pv_to_w([p_from_u(2),p_from_u(1)])
_pv_to_w([p_from_u(2),p_from_u(0)])
_pv_to_w(LABEL_RANGES["PV"][0])
# _w_to_pv_intervals(_pv_to_w(LABEL_RANGES["PV"][0], w_sign=0))
# _w_to_pv_intervals(_pv_to_w(LABEL_RANGES["PV"][1]))
# _w_to_pv_intervals(_pv_to_w(LABEL_RANGES["PV"][2]))
# _w_to_pv_intervals(_pv_to_w(LABEL_RANGES["PV"][3]))
# _w_to_pv_intervals(_pv_to_w(LABEL_RANGES["PV"][4]))

print(_pv_to_w(LABEL_RANGES["PV"][1], w_sign=0))
simplify_intervals(_pv_to_w(LABEL_RANGES["PV"][1], w_sign=0))

[[-0.1333333333333333, -0.04999999999999997], [0.04999999999999997, 0.1333333333333333]]


[[-0.1333, -0.05], [0.05, 0.1333]]

In [115]:
demo_tree = {
    0: {  # depth label 0 (surface)
        0: {  # PV valleys
            (1, 2): {  # erosion labels 1-2
                (1, 2): {  # continentalness labels
                    "*": {
                        "*": {
                            "biome": "minecraft:snowy_taiga"
                        }
                    }
                }
            }
        },
        (3, 4): {  # PV high/peaks
            (0, 1): {           # erosion 0-1
                (5, 6): {       # continentalness mid/far inland
                    4: {        # temperature hot
                        "*": {
                           0: { # weirdness 0
                               "biome": "minecraft:badlands"
                           },
                           1: { # weirdness 1
                               "biome": "minecraft:desert"
                           }
                        }
                    }
                }
            }
        }
    }
}

demo_tree_2 = {
    (1, 2): {  # depth labels shallow-mid caves
        "biome": "minecraft:lush_caves"
    },
    (3, 4): {  # depth labels deep-mid caves
        "biome": "minecraft:dripstone_caves"
    }
}

# Generation

## Tree

In [140]:
# -----------------------------------------------------------
#  Example tree spec
# -----------------------------------------------------------
# order: depth, PV, erosion, continentalness, temperature, humidity, weirdness

# depth: 0-4
# PV: 0-4
# erosion: 0-6
# continentalness: 0-8
# temperature: 0-5
# humidity: 0-6
# weirdness: 0-1

ocean_tree = {
    0: {
        "*": {
            "*": {
                0: { # todo: mushroom fields, but we can make it volcanic
                    "biome": "minecraft:deep_frozen_ocean"
                },
                1: {
                    "biome": "minecraft:deep_frozen_ocean"
                },
                2: {
                    "biome": "minecraft:frozen_ocean"
                }
            }
        }
    }
}

#   D   PV     Ero     cont      temp    hum   weirdness
underground_tree = {
    1: {'*': {   '*': {(6, 8): {(0, 3): {'*': {'biome': 'the_winter_rescue:andesite_caves'}},
                                (4, 5): {'*': {'biome': 'the_winter_rescue:mycelium_caves'}}}},
              (0, 3): {(3, 5): {(0, 1): {'*': {'biome': 'the_winter_rescue:ice_caves'}},
                                (2, 5): {'*': {'biome': 'minecraft:dripstone_caves'}}}},
              (4, 6): {(3, 5): {(0, 5): {'*': {'biome': 'the_winter_rescue:brine_deposits'}}}}}},
    
    2: {'*': {   '*': {(6, 8): {(0, 3): {'*': {'biome': 'the_winter_rescue:andesite_caves'}},
                                (4, 5): {'*': {'biome': 'the_winter_rescue:mycelium_caves'}}}},
              (0, 3): {(3, 5): {(0, 1): {'*': {'biome': 'the_winter_rescue:ice_caves'}},
                                (2, 5): {'*': {'biome': 'minecraft:dripstone_caves'}}}},
              (4, 6): {(3, 5): {(0, 5): {'*': {'biome': 'the_winter_rescue:brine_deposits'}}}}}},
    
    # deep-ocean hydrothermal deposits
    3: {'*': {   '*': {
                       (3, 8): {   '*': {'*': {'biome': 'the_winter_rescue:darkfang_caves'}}},
                       (0, 2): {(0, 2): {'*': {'biome': 'the_winter_rescue:darkfang_caves'}},
                                (3, 3): {'*': {'biome': 'the_winter_rescue:diorite_caves'}},
                                (4, 4): {'*': {'biome': 'minecraft:lush_caves'}},
                                (5, 5): {'*': {'biome': 'the_winter_rescue:hydrothermal_deposits'}},
                                (6, 6): {'*': {'biome': 'the_winter_rescue:crust_chasms'}}
                                }
                       },
              }
        },
    
    # todo: change andesite caves to basalt caves
    4: {'*': {
            #   # high erosion, hydrothermal deposits dominant in ocean and coast
            #   (4, 6): {
            #            (4, 8): {   '*': {'*': {'biome': 'the_winter_rescue:darkfang_caves'}}},
            #            (0, 3): {(0, 3): {'*': {'biome': 'the_winter_rescue:andesite_caves'}},
            #                     (3, 3): {'*': {'biome': 'minecraft:lush_caves'}},
            #                     (4, 5): {'*': {'biome': 'the_winter_rescue:hydrothermal_deposits'}}}
            #            },
              # low erosion, results in magmatic deposits
              (0, 6): {(6, 8): {(0, 2): {'*': {'biome': 'the_winter_rescue:darkfang_caves'}},
                                (3, 3): {'*': {'biome': 'the_winter_rescue:andesite_caves'}},
                                (4, 4): {'*': {'biome': 'minecraft:lush_caves'}},
                                (5, 5): {'*': {'biome': 'the_winter_rescue:magmatic_deposits'}},
                                (6, 6): {'*': {'biome': 'the_winter_rescue:crust_chasms'}}
                                },
                       (3, 5): {   '*': {'*': {'biome': 'the_winter_rescue:darkfang_caves'}}},
                       (0, 2): {(0, 2): {'*': {'biome': 'the_winter_rescue:darkfang_caves'}},
                                (3, 3): {'*': {'biome': 'the_winter_rescue:diorite_caves'}},
                                (4, 4): {'*': {'biome': 'minecraft:lush_caves'}},
                                (5, 5): {'*': {'biome': 'the_winter_rescue:hydrothermal_deposits'}},
                                (6, 6): {'*': {'biome': 'the_winter_rescue:crust_chasms'}}
                                }
                       },
            #   # least erosion, results in crust chasms
            #   (0, 1): {(6, 8): {(0, 2): {'*': {'biome': 'the_winter_rescue:andesite_caves'}},
            #                     (3, 3): {'*': {'biome': 'minecraft:lush_caves'}},
            #                     (4, 5): {'*': {'biome': 'the_winter_rescue:crust_chasms'}}},
            #            (2, 5): {   '*': {'*': {'biome': 'the_winter_rescue:darkfang_caves'}}},
            #            (0, 1): {(0, 3): {'*': {'biome': 'the_winter_rescue:andesite_caves'}},
            #                     (3, 3): {'*': {'biome': 'minecraft:lush_caves'}},
            #                     (4, 5): {'*': {'biome': 'the_winter_rescue:hydrothermal_deposits'}}}
            #            },
              }
        }
}

surface_tree = {
    0: {
        "*": {
            "*": {
                (3, 8): {
                    "biome": "minecraft:snowy_taiga"
                }
            }
        }
    }
}


# -----------------------------------------------------------
#  Convert tree → JSON
# -----------------------------------------------------------

ocean_spec = tree_to_spec(ocean_tree)
underground_spec = tree_to_spec(underground_tree)
surface_spec = tree_to_spec(surface_tree)
ocean_biome_rows = convert(ocean_spec)
underground_biome_rows = convert(underground_spec)
surface_biome_rows = convert(surface_spec)
biome_rows = ocean_biome_rows + underground_biome_rows + surface_biome_rows
preset_json = to_preset_file(biome_rows)

out_path = "tree_normal.json"
with open(out_path, "w") as f:
    json.dump(preset_json, f, indent=2)
print("Wrote", out_path)

Wrote tree_normal.json


## List

In [141]:
# --------------------------------------------------------------------
# 3.  Demo with 3 biomes to illustrate the format
# --------------------------------------------------------------------

example_spec = {
    # oceans
    # deep frozen ocean: deep, cold, far from land
    "minecraft:deep_frozen_ocean": {
        "continentalness": 1
    },
    # frozen ocean: deep, cold, close to land
    "minecraft:frozen_ocean": {
        "continentalness": 2
    },
    
    ### CAVES
        
    # depth 4
    # deepslate: deep
    "the_winter_rescue:darkfang_caves": {
        "depth": 4,
        "continentalness": (3, 5),
        "temperature": (0, 3)
    },
    # lush caves: warm, deep
    "minecraft:lush_caves": {
        "depth": 4,
        "continentalness": (3, 5),
        "temperature": (4, 5)
    },
    # hydrothermal_deposits: deep, warm
    "the_winter_rescue:hydrothermal_deposits": {
        "depth": 4,
        "continentalness": 6,
        "temperature": (4, 5)
    },
    "the_winter_rescue:darkfang_caves": {
        "depth": 4,
        "continentalness": 6,
        "temperature": (0, 3)
    },
    # magmatic deposits: deep, hot
    "the_winter_rescue:magmatic_deposits": {
        "depth": 4,
        "continentalness": 7,
        "temperature": (4, 5)
    },
    "the_winter_rescue:darkfang_caves": {
        "depth": 4,
        "continentalness": 7,
        "temperature": (0, 3)
    },
    "the_winter_rescue:crust_chasms": {
        "depth": 4,
        "continentalness": 8,
        "temperature": (4, 5)
    },
    "the_winter_rescue:darkfang_caves": {
        "depth": 4,
        "continentalness": 8,
        "temperature": (0, 3)
    },
    
    # depth 3
    # intrusive igneous: deep
    "the_winter_rescue:diorite_caves": {
        "depth": 3,
        "continentalness": (3, 5)
    },
    # hydrothermal_deposits: deep, warm
    "the_winter_rescue:hydrothermal_deposits": {
        "depth": 3,
        "continentalness": (6, 8),
        "temperature": (4, 5)
    },
    "the_winter_rescue:darkfang_caves": {
        "depth": 4,
        "continentalness": (6, 8),
        "temperature": (0, 3)
    },
    
    # depth 2
    
    # sedimentary: shallow, closer to ocean
    "minecraft:dripstone_caves": {
        "depth": 2,
        "continentalness": (3, 5),
        "erosion": (0, 3),
        "temperature": (2, 5),
    },
    "the_winter_rescue:ice_caves": {
        "depth": 2,
        "continentalness": (3, 5),
        "erosion": (0, 3),
        "temperature": (0, 1),
    },
    # brine_deposits: close to ocean, shallow, erosion
    "the_winter_rescue:brine_deposits": {
        "depth": 2,
        "continentalness": (3, 5),
        "erosion": (4, 6),
        "temperature": (0, 5),
    },
    # extrusive igneous: shallow
    "the_winter_rescue:andesite_caves": {
        "depth": 2,
        "continentalness": (6, 8),
        "temperature": (0, 3),
    },
    "the_winter_rescue:mycelium_caves": {
        "depth": 2,
        "continentalness": (6, 8),
        "temperature": (4, 5)
    },
    
    # depth 1
    
    # sedimentary: shallow, closer to ocean
    "minecraft:dripstone_caves": {
        "depth": 1,
        "continentalness": (3, 5),
        "erosion": (0, 3),
        "temperature": (2, 5),
    },
    "the_winter_rescue:ice_caves": {
        "depth": 1,
        "continentalness": (3, 5),
        "erosion": (0, 3),
        "temperature": (0, 1),
    },
    # brine_deposits: close to ocean, shallow, erosion
    "the_winter_rescue:brine_deposits": {
        "depth": 1,
        "continentalness": (3, 5),
        "erosion": (4, 6),
        "temperature": (0, 5),
    },
    # extrusive igneous: shallow
    "the_winter_rescue:andesite_caves": {
        "depth": 1,
        "continentalness": (6, 8),
        "temperature": (0, 3),
    },
    "the_winter_rescue:mycelium_caves": {
        "depth": 1,
        "continentalness": (6, 8),
        "temperature": (4, 5)
    },  
    
    
    ### SURFACE
    
    ## PV: Valleys. TOOD: reimplment how valleys are from the weirdness. 
    
    # Erosion: 0-1
    "minecraft:frozen_river": {
        "depth": 0,
        "erosion": (0, 1),
        "continentalness": (3, 4)
    },
    
    # middle biomes: divided by humidity and temperature
    
    # T = 0
    "minecraft:snowy_plains": {
        "depth": 0,
        "erosion": (0, 1),
        "continentalness": (5, 8),
        "temperature": 0,
        "humidity": 0,
        "weirdness": (0, 2)
    },
    "minecraft:ice_spikes": {
        "depth": 0,
        "erosion": (0, 1),
        "continentalness": (5, 8),
        "temperature": 0,
        "humidity": 0,
        "weirdness": (3, 4)
    },
    "minecraft:snowy_plains": {
        "depth": 0,
        "erosion": (0, 1),
        "continentalness": (5, 8),
        "temperature": 0,
        "humidity": 1
    },
    "minecraft:snowy_plains": {
        "depth": 0,
        "erosion": (0, 1),
        "continentalness": (5, 8),
        "temperature": 0,
        "humidity": 2,
        "weirdness": (0, 2)
    },
    "the_winter_rescue:nature/snowy_shrubland": {
        "depth": 0,
        "erosion": (0, 1),
        "continentalness": (5, 8),
        "temperature": 0,
        "humidity": 2,
        "weirdness": (3, 4)
    },
    "minecraft:snowy_taiga": {
        "depth": 0,
        "erosion": (0, 1),
        "continentalness": (5, 8),
        "temperature": 0,
        "humidity": (3, 4)
    },
    
    # T = 1
    "minecraft:snowy_plains": {
        "depth": 0,
        "erosion": (0, 1),
        "continentalness": (5, 8),
        "temperature": 1,
        "humidity": (0, 1)
    },
    "the_winter_rescue:destroyed_forest": {
        "depth": 0,
        "erosion": (0, 1),
        "continentalness": (5, 8),
        "temperature": 1,
        "humidity": 2
    },
    "the_winter_rescue:nature/snowy_shrubland": {
        "depth": 0,
        "erosion": (0, 1),
        "continentalness": (5, 8),
        "temperature": 1,
        "humidity": 3
    },
    "minecraft:snowy_taiga": {
        "depth": 0,
        "erosion": (0, 1),
        "continentalness": (5, 8),
        "temperature": 1,
        "humidity": 4
    },
    
    # T = 2
    "minecraft:snowy_plains": {
        "depth": 0,
        "erosion": (0, 1),
        "continentalness": (5, 8),
        "temperature": 2,
        "humidity": (0, 1)
    },
    "the_winter_rescue:destroyed_birch_forest": {
        "depth": 0,
        "erosion": (0, 1),
        "continentalness": (5, 8),
        "temperature": 2,
        "humidity": 2
    },
    "the_winter_rescue:nature/snowy_shrubland": {
        "depth": 0,
        "erosion": (0, 1),
        "continentalness": (5, 8),
        "temperature": 2,
        "humidity": 3
    },
    "minecraft:snowy_taiga": {
        "depth": 0,
        "erosion": (0, 1),
        "continentalness": (5, 8),
        "temperature": 2,
        "humidity": 4
    },
    
    # T = 3
    # todo: destroyed_savanna
    "the_winter_rescue:nature/snowy_shrubland": {
        "depth": 0,
        "erosion": (0, 1),
        "continentalness": (5, 8),
        "temperature": 3,
        "humidity": (0, 1)
    },
    "minecraft:snowy_taiga": {
        "depth": 0,
        "erosion": (0, 1),
        "continentalness": (5, 8),
        "temperature": 2,
        "humidity": 2
    },
    "the_winter_rescue:nature/frostbough_forest": {
        "depth": 0,
        "erosion": (0, 1),
        "continentalness": (5, 8),
        "temperature": 3,
        "humidity": 3
    },
    "the_winter_rescue:nature/ironwinter_hollow": {
        "depth": 0,
        "erosion": (0, 1),
        "continentalness": (5, 8),
        "temperature": 3,
        "humidity": 4
    },
    
    
    # badlands: high temp
    "minecraft:badlands": {
        "depth": 0,
        "erosion": (0, 1),
        "continentalness": (5, 8),
        "temperature": (4, 5)
    },
    
    # Erosion: 2-5
    "minecraft:frozen_river": {
        "depth": 0,
        "erosion": (2, 5),
        "continentalness": (3, 8)
    },
    
    # Erosion: 6
    "minecraft:frozen_river": {
        "depth": 0,
        "erosion": 6,
        "continentalness": 3
    },
    "minecraft:frozen_river": {
        "depth": 0,
        "erosion": 6,
        "continentalness": (4, 8),
        "temperature": (0, 2)
    },
    "the_winter_rescue:nature/destroyed_marsh": {
        "depth": 0,
        "erosion": 6,
        "continentalness": (4, 8),
        "temperature": (3, 5)
    }
    
    ## PV: Low.
    
    ## PV: Mid.
    
    ## PV: High.
    
    ## PV: Peaks.
    
    
    # TODO: find a way to generate middle biomes / badland biomes / plataeu biomes given certain set of params more quickly.

    
}

converted = convert_biome_specs(example_spec)

out_path = "new_normal.json"
with open(out_path, "w") as f:
    json.dump(converted, f, indent=2)

out_path

IndexError: list index out of range